#Notebook para  Capa Silver

In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
%sql
create widget text storageName default "adventureworksltadlsd";
CREATE WIDGET TEXT catalogName DEFAULT 'catadvwd';
CREATE WIDGET TEXT bronzeSchemaName DEFAULT 'bronze';
CREATE WIDGET TEXT silverSchemaName DEFAULT 'silver';


In [0]:
catalogo = dbutils.widgets.get("catalogName")
bronzeSchema = dbutils.widgets.get("bronzeSchemaName")
silverSchema = dbutils.widgets.get("silverSchemaName")

In [0]:
def merge_to_silver(df, nombre_tabla, llave_negocio):
    """
    Aplica upsert (MERGE) de un DataFrame hacia una tabla Silver.
 
    Parámetros:
        df (DataFrame): datos ya limpios y validados, listos para Silver.
        nombre_tabla (str): nombre de la tabla destino (sin catálogo/esquema).
        llave_negocio (str): columna usada para hacer match entre origen y destino.
    """
    tabla_destino = f"{catalogo}.{silverSchema}.{nombre_tabla}"
 
    try:
        if spark.catalog.tableExists(tabla_destino):
            delta_tabla = DeltaTable.forName(spark, tabla_destino)
            (delta_tabla.alias("destino")
                .merge(df.alias("origen"), f"destino.{llave_negocio} = origen.{llave_negocio}")
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute())
            print(f" MERGE aplicado sobre {tabla_destino} (llave: {llave_negocio})")
        else:
            df.write.format("delta").mode("overwrite").saveAsTable(tabla_destino)
            print(f" Tabla {tabla_destino} creada por primera vez ({df.count()} filas)")
    except Exception as e:
        print(f" Error al cargar {tabla_destino}: {e}")
        raise

In [0]:
df_customer_bronze = spark.table(f"{catalogo}.{bronzeSchema}.customers")

In [0]:

filas_totales = df_customer_bronze.count()

In [0]:
df_customer_silver = df_customer_bronze.filter(
    col("firstName").isNotNull() & col("lastName").isNotNull()
)

In [0]:
filas_validas = df_customer_silver.count()
print(f"Customer -> total: {filas_totales} | válidas: {filas_validas} | descartadas: {filas_totales - filas_validas}")

Customer -> total: 847 | válidas: 847 | descartadas: 0


In [0]:
merge_to_silver(df_customer_silver, "customers", "customerId")

 Tabla catadvwd.silver.customers creada por primera vez (847 filas)


In [0]:
df_product_bronze = spark.table(f"{catalogo}.{bronzeSchema}.productos")

In [0]:
filas_totales = df_product_bronze.count()

In [0]:
df_product_silver = df_product_bronze.filter(col("productId").isNotNull())

In [0]:
filas_validas = df_product_silver.count()
print(f"Product -> total: {filas_totales} | válidas: {filas_validas} | descartadas: {filas_totales - filas_validas}")

Product -> total: 299 | válidas: 295 | descartadas: 4


In [0]:
merge_to_silver(df_product_silver, "products", "productId")

 Tabla catadvwd.silver.products creada por primera vez (295 filas)


In [0]:

df_sod_bronze = spark.table(f"{catalogo}.{bronzeSchema}.salesorderdetail")
 

In [0]:
filas_totales = df_sod_bronze.count()
 
# Paso 1: completitud
df_sod_completo = df_sod_bronze.filter(
    col("salesOrderId").isNotNull() &
    col("salesOrderDetailId").isNotNull() &
    col("orderQty").isNotNull() &
    col("productId").isNotNull() &
    col("unitPrice").isNotNull()
)
 

In [0]:
# Paso 2: unicidad de salesOrderDetailId (nos quedamos con el registro más reciente)
ventana_unicidad = Window.partitionBy("salesOrderDetailId").orderBy(col("modifiedDate").desc_nulls_last())
 
df_sod_unico = (df_sod_completo
    .withColumn("_rn", row_number().over(ventana_unicidad))
    .filter(col("_rn") == 1)
    .drop("_rn")
)
 

In [0]:
# Paso 3: rowguid no se mapea a Silver
df_salesorderdetail_silver = df_sod_unico.drop("rowguid")
 
filas_validas = df_salesorderdetail_silver.count()
print(f"SalesOrderDetail -> total: {filas_totales} | válidas y únicas: {filas_validas} | descartadas: {filas_totales - filas_validas}")
 

SalesOrderDetail -> total: 542 | válidas y únicas: 542 | descartadas: 0


In [0]:
merge_to_silver(df_salesorderdetail_silver, "salesorderdetail", "salesOrderDetailId")

 Tabla catadvwd.silver.salesorderdetail creada por primera vez (542 filas)


In [0]:
df_soh_bronze = spark.table(f"{catalogo}.{bronzeSchema}.salesorderheader")
 
filas_totales = df_soh_bronze.count()
 
df_soh_completo = df_soh_bronze.filter(col("customerId").isNotNull())
 
df_salesorderheader_silver = df_soh_completo.drop("rowguid")
 
filas_validas = df_salesorderheader_silver.count()
print(f"SalesOrderHeader -> total: {filas_totales} | válidas: {filas_validas} | descartadas: {filas_totales - filas_validas}")

SalesOrderHeader -> total: 32 | válidas: 32 | descartadas: 0


In [0]:
merge_to_silver(df_salesorderheader_silver, "salesorderheader", "salesOrderId")

 Tabla catadvwd.silver.salesorderheader creada por primera vez (32 filas)


In [0]:
for tabla in ["customers", "products", "salesorderdetail", "salesorderheader"]:
    conteo = spark.table(f"{catalogo}.{silverSchema}.{tabla}").count()
    print(f"{catalogo}.{silverSchema}.{tabla}: {conteo} filas")
 

catadvwd.silver.customers: 847 filas
catadvwd.silver.products: 295 filas
catadvwd.silver.salesorderdetail: 542 filas
catadvwd.silver.salesorderheader: 32 filas
